# LLM Labeling for Ground Truth

This notebook labels the shuffled candidate pool produced by `1_build_groundtruth.ipynb`.

The labeling strategy is query-batched:

1. Load `blinded_annotation_items.jsonl`.
2. Group candidates by `query_id`.
3. Send one query and all of its candidate documents to the LLM.
4. Parse the returned JSON labels.
5. Save flattened pair-level labels to `llm_groundtruth_labels.jsonl`.

With 500 queries, this means about 500 LLM calls rather than one call per query-document pair.

## 1. Imports and Environment Loading

In [1]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Optional

import pandas as pd

In [2]:
def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


def find_repo_root(finalproject_root: Path) -> Path:
    """Return the repository root that contains Finalproject."""
    return finalproject_root.parent


def load_env_file(env_path: Path) -> None:
    """Load key=value pairs from a .env file without overriding existing environment variables."""
    if not env_path.exists():
        print(f"No .env file found at: {env_path}")
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


def get_env_int(name: str, default: int) -> int:
    value = os.getenv(name)
    return default if value in (None, "") else int(value)


def get_env_float(name: str, default: float) -> float:
    value = os.getenv(name)
    return default if value in (None, "") else float(value)


FINALPROJECT_ROOT = find_finalproject_root()
REPO_ROOT = find_repo_root(FINALPROJECT_ROOT)
ENV_PATH = REPO_ROOT / ".env"
load_env_file(ENV_PATH)

DATA_PATH = FINALPROJECT_ROOT / "data" / "all_recipes_final.csv"
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
BLINDED_ANNOTATION_ITEMS_PATH = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"

print("Finalproject root:", FINALPROJECT_ROOT)
print(".env path:", ENV_PATH)
print("Blinded annotation items:", BLINDED_ANNOTATION_ITEMS_PATH)

Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
.env path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\.env
Blinded annotation items: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\blinded_annotation_items.jsonl


## 2. Main Configuration

In [3]:
# -------------------------------------------------------
# OpenAI-compatible LiteLLM labeling config
# -------------------------------------------------------
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://litellm.imt-soft/v1")
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "gpt-4o-mini")
LLM_API_KEY = os.getenv("LLM_API_KEY")

# Self-hosted endpoint defaults: generous response budget and timeout.
LLM_TEMPERATURE = get_env_float("LLM_TEMPERATURE", 0.0)
LLM_MAX_TOKENS = get_env_int("LLM_MAX_TOKENS", 16384)
LLM_REQUEST_TIMEOUT_SECONDS = get_env_int("LLM_REQUEST_TIMEOUT_SECONDS", 300)

if not LLM_API_KEY:
    raise ValueError("LLM_API_KEY is required for the OpenAI-compatible LiteLLM endpoint.")
LLM_MAX_RETRIES = get_env_int("LLM_MAX_RETRIES", 5)

# None means label all query groups. Set a small integer for debugging.
MAX_QUERY_GROUPS_TO_LABEL = None

RESUME_LLM_LABELING = True
LLM_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"
LLM_QUERY_LOGS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_query_logs.jsonl"


def normalize_openai_compatible_base_url(raw_base_url: str) -> str:
    """Normalize LiteLLM proxy URL for OpenAI-compatible SDK clients."""
    base_url = str(raw_base_url).strip().rstrip("/")
    if not base_url:
        raise ValueError("LLM_BASE_URL cannot be empty.")
    if not base_url.startswith(("http://", "https://")):
        base_url = "https://" + base_url
    if not base_url.endswith("/v1"):
        base_url = base_url + "/v1"
    return base_url


LLM_BASE_URL = normalize_openai_compatible_base_url(LLM_BASE_URL)

print("LLM base URL:", LLM_BASE_URL)
print("LLM model:", LLM_MODEL_NAME)
print("Pair-level labels output:", LLM_LABELS_PATH)
print("Query-level logs output:", LLM_QUERY_LOGS_PATH)

LLM base URL: https://litellm.imt-soft.com/v1
LLM model: local-std-03
Pair-level labels output: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\llm_groundtruth_labels.jsonl
Query-level logs output: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\rec_and_eval\groundtruth_outputs\annotation\llm_groundtruth_query_logs.jsonl


## 3. Load Shuffled Annotation Items

In [4]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if line.strip():
                records.append(json.loads(line))
    return records


def append_jsonl_record(output_path: Path, record: dict) -> None:
    """Append one JSON-serializable record to a JSONL file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("a", encoding="utf-8") as output_file:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()


def compact_text(value: str, max_characters: int) -> str:
    """Clip long text fields so one query prompt remains manageable."""
    text = str(value or "").strip()
    if len(text) > max_characters:
        return text[: max_characters - 3].rstrip() + "..."
    return text




def build_recipe_description_lookup(data_path: Path) -> dict[int, str]:
    """Build doc_id -> recipe description lookup for older annotation files.

    New blinded annotation files already contain recipe_description. This helper
    keeps the labeling notebooks compatible with existing JSONL files that were
    exported before description was included.
    """
    recipe_dataframe = pd.read_csv(data_path).reset_index(drop=True)
    if "doc_id" not in recipe_dataframe.columns:
        recipe_dataframe.insert(0, "doc_id", recipe_dataframe.index.astype(int))
    if "description" not in recipe_dataframe.columns:
        return {}

    return {
        int(row["doc_id"]): compact_text(row.get("description", ""), max_characters=700)
        for _, row in recipe_dataframe[["doc_id", "description"]].iterrows()
    }


def add_missing_recipe_descriptions_to_annotation_items(
    annotation_items: list[dict],
    description_lookup: dict[int, str],
) -> list[dict]:
    """Attach recipe_description when an existing blinded JSONL file does not have it."""
    enriched_items = []
    for annotation_item in annotation_items:
        enriched_item = dict(annotation_item)
        if not str(enriched_item.get("recipe_description", "")).strip():
            doc_id = int(enriched_item["doc_id"])
            enriched_item["recipe_description"] = description_lookup.get(doc_id, "")
        enriched_items.append(enriched_item)
    return enriched_items

def group_annotation_items_by_query(annotation_items: list[dict]) -> list[dict]:
    """Group shuffled annotation items into one prompt payload per query."""
    grouped: dict[int, dict] = {}
    for item in annotation_items:
        query_id = int(item["query_id"])
        grouped.setdefault(
            query_id,
            {
                "query_id": query_id,
                "query_text": str(item["query_text"]),
                "documents": [],
            },
        )
        grouped[query_id]["documents"].append(item)

    query_groups = []
    for query_group in grouped.values():
        query_group["documents"] = sorted(
            query_group["documents"],
            key=lambda item: int(item["blinded_position"]),
        )
        query_groups.append(query_group)
    return sorted(query_groups, key=lambda group: group["query_id"])


def build_candidate_id(blinded_position: int) -> str:
    """Build a short stable candidate ID for the LLM to copy."""
    return f"D{int(blinded_position):03d}"


def build_document_payload(annotation_item: dict) -> dict:
    """Build the full internal document object for one candidate."""
    blinded_position = int(annotation_item["blinded_position"])
    recipe_description = (
        annotation_item.get("recipe_description", "")
        or annotation_item.get("description", "")
    )
    return {
        "candidate_id": build_candidate_id(blinded_position),
        "blinded_position": blinded_position,
        "doc_id": int(annotation_item["doc_id"]),
        "title": compact_text(annotation_item.get("recipe_title", ""), max_characters=180),
        "recipe_type": compact_text(annotation_item.get("recipe_type", ""), max_characters=80),
        "description": compact_text(recipe_description, max_characters=700),
    }


def build_prompt_document(document: dict) -> dict:
    """Build the document object shown to the LLM.

    doc_id is intentionally hidden from the prompt. The LLM only has to copy a short
    candidate_id, and the code maps candidate_id back to doc_id after parsing.
    """
    return {
        "candidate_id": document["candidate_id"],
        "title": document["title"],
        "recipe_type": document["recipe_type"],
        "description": document["description"],
    }


def build_query_payload(query_group: dict) -> dict:
    """Build the full query-level labeling payload."""
    documents = [build_document_payload(item) for item in query_group["documents"]]
    return {
        "query_id": int(query_group["query_id"]),
        "query_text": str(query_group["query_text"]),
        "documents": documents,
        "prompt_documents": [build_prompt_document(document) for document in documents],
    }


def strip_markdown_json_fence(raw_text: str) -> str:
    """Remove common markdown code fences around JSON output."""
    text = str(raw_text).strip()
    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    return fenced_match.group(1).strip() if fenced_match else text


def parse_query_label_response(raw_response_text: str, expected_payload: dict) -> list[dict]:
    """Parse and validate LLM labels for one query using candidate_id.

    The LLM is only expected to copy candidate_id values such as D001, D002, ...
    doc_id is recovered by this parser from the original payload.
    """
    cleaned_response = strip_markdown_json_fence(raw_response_text)
    parsed_response = json.loads(cleaned_response)

    if not isinstance(parsed_response, dict):
        raise ValueError("LLM response must be a JSON object.")
    if int(parsed_response.get("query_id")) != int(expected_payload["query_id"]):
        raise ValueError(
            f"LLM response query_id {parsed_response.get('query_id')} does not match expected {expected_payload['query_id']}."
        )

    labels = parsed_response.get("labels")
    if not isinstance(labels, list):
        raise ValueError("LLM response must contain a labels list.")

    expected_by_candidate_id = {
        str(document["candidate_id"]): document
        for document in expected_payload["documents"]
    }
    expected_candidate_ids = set(expected_by_candidate_id.keys())

    parsed_records = []
    seen_candidate_ids = set()
    unexpected_candidate_ids = []

    for label_item in labels:
        candidate_id = str(label_item.get("candidate_id", "")).strip()
        relevance = int(label_item["relevance"])
        if relevance not in {0, 1, 2, 3}:
            raise ValueError(f"Invalid relevance label {relevance} for candidate_id={candidate_id}.")
        if candidate_id not in expected_by_candidate_id:
            unexpected_candidate_ids.append(candidate_id)
            continue
        if candidate_id in seen_candidate_ids:
            raise ValueError(f"Duplicate label for candidate_id={candidate_id}.")
        seen_candidate_ids.add(candidate_id)

        expected_document = expected_by_candidate_id[candidate_id]
        parsed_records.append(
            {
                "query_id": int(expected_payload["query_id"]),
                "candidate_id": candidate_id,
                "doc_id": int(expected_document["doc_id"]),
                "blinded_position": int(expected_document["blinded_position"]),
                "relevance": relevance,
                "label_match_strategy": "candidate_id",
                "label_reconciled": False,
            }
        )

    if unexpected_candidate_ids:
        raise ValueError(
            "LLM returned unknown candidate_id values: "
            f"{unexpected_candidate_ids[:10]}. Expected IDs include: {sorted(expected_candidate_ids)[:10]}"
        )

    missing_candidate_ids = expected_candidate_ids - seen_candidate_ids
    if missing_candidate_ids:
        raise ValueError(
            f"LLM response missed {len(missing_candidate_ids)} candidates: "
            f"{sorted(missing_candidate_ids)[:10]}"
        )

    return sorted(parsed_records, key=lambda record: record["blinded_position"])


In [5]:
blinded_annotation_items = load_jsonl_records(BLINDED_ANNOTATION_ITEMS_PATH)
recipe_description_lookup = build_recipe_description_lookup(DATA_PATH)
blinded_annotation_items = add_missing_recipe_descriptions_to_annotation_items(
    annotation_items=blinded_annotation_items,
    description_lookup=recipe_description_lookup,
)
query_groups = group_annotation_items_by_query(blinded_annotation_items)

print("Number of blinded annotation items:", len(blinded_annotation_items))
print("Number of query groups:", len(query_groups))
print("Candidates per query:")
print(pd.Series([len(group["documents"]) for group in query_groups]).describe())

query_groups[0]["query_id"], query_groups[0]["query_text"], len(query_groups[0]["documents"])

Number of blinded annotation items: 25000
Number of query groups: 500
Candidates per query:
count    500.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
dtype: float64


(0, 'Bánh trung thu nướng nhân đậu xanh bằng nồi chiên không dầu', 50)

## 4. Query-Batched Relevance Prompt

In [6]:
LLM_JUDGE_PROMPT_VERSION = "recipe_relevance_query_batch_title_type_description_v3"

RELEVANCE_JUDGE_SYSTEM_PROMPT = """You are an expert relevance assessor for a Vietnamese recipe search system.
Your task is to label how useful each candidate recipe would be for the user query.
Use only the provided title, recipe_type, and description fields.
Prefer semantic usefulness over exact word matching.
Return valid JSON only. Do not include markdown, comments, or explanations.
"""

RELEVANCE_JUDGE_USER_PROMPT_TEMPLATE = """TASK
Label every candidate document for the given query using a 0-3 graded relevance scale.

IMPORTANT ID RULE
- Each candidate has a short candidate_id such as D001, D002, D003.
- You MUST copy candidate_id values exactly from CANDIDATE DOCUMENTS.
- Do NOT invent candidate_id values.
- Do NOT output doc_id.
- Do NOT output blinded_position.
- Output exactly one label for every candidate_id in CANDIDATE DOCUMENTS.

AVAILABLE EVIDENCE
- Each candidate contains only title, recipe_type, and description.
- Treat title as the strongest signal.
- Use recipe_type and description as supporting signals.
- Do not require evidence outside these three fields.

QUERY
query_id: {query_id}
query_text: {query_text}

CANDIDATE DOCUMENTS
{documents_json}

RELEVANCE DEFINITIONS
3 = Highly relevant:
- The candidate directly satisfies the query intent, or is a natural recipe variant that a user would reasonably accept.
- Exact wording is not required when the title/type/description clearly refer to the same dish, drink, dessert, topping, flavor family, or recipe family.

2 = Relevant:
- The candidate is not the exact requested item, but it is a close substitute or close variant in the same recipe family.
- It would still be useful to show for the query because most of the food/drink intent is preserved.

1 = Somewhat relevant:
- The candidate is related only at a broad level, such as the same general category, occasion, flavor direction, or food/drink group.
- It may be useful for exploration, but it is not a close answer to the query.

0 = Not relevant:
- The candidate would not substantially help the user query.
- Generic Vietnamese food words, generic adjectives, or weak surface overlap are not enough.

CALIBRATION GUIDANCE
- If a candidate is a clear member of the same dish/drink/dessert family as the query, avoid labeling it 0.
- If unsure between two adjacent labels, choose the label that best reflects whether the result would be useful to a real recipe-search user.
- Keep the full 0-3 scale meaningful: use 3 for strong matches, 2 for close variants, 1 for loose related items, and 0 for unrelated items.

MANDATORY OUTPUT RULES
- Return JSON only.
- Return exactly this top-level shape:
{{
  "query_id": {query_id},
  "labels": [
    {{"candidate_id": "D001", "relevance": 0}},
    {{"candidate_id": "D002", "relevance": 3}}
  ]
}}
- The labels list length MUST equal the number of candidate documents.
- Every candidate_id from CANDIDATE DOCUMENTS must appear exactly once.
- relevance must be one of: 0, 1, 2, 3.
"""


def build_relevance_judge_prompt_for_query(query_group: dict) -> dict:
    """Build one prompt for one query and all its candidate documents."""
    payload = build_query_payload(query_group)
    documents_json = json.dumps(payload["prompt_documents"], ensure_ascii=False, indent=2)
    user_prompt = RELEVANCE_JUDGE_USER_PROMPT_TEMPLATE.format(
        query_id=payload["query_id"],
        query_text=payload["query_text"],
        documents_json=documents_json,
    )
    return {
        "system_prompt": RELEVANCE_JUDGE_SYSTEM_PROMPT,
        "user_prompt": user_prompt,
        "prompt_version": LLM_JUDGE_PROMPT_VERSION,
        "payload": payload,
    }


## 5. OpenAI-Compatible LiteLLM Caller

In [7]:
def call_openai_compatible_query_judge(system_prompt: str, user_prompt: str) -> str:
    """Call the LiteLLM OpenAI-compatible endpoint and return raw text for one query group."""
    try:
        from openai import OpenAI
    except ImportError as import_error:
        raise ImportError("OpenAI SDK is not installed. Install it with `pip install openai`.") from import_error

    client = OpenAI(
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        timeout=LLM_REQUEST_TIMEOUT_SECONDS,
    )

    last_error = None
    for attempt_index in range(1, LLM_MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS,
            )
            return response.choices[0].message.content.strip()
        except Exception as request_error:
            last_error = request_error
            if attempt_index >= LLM_MAX_RETRIES:
                break
            sleep_seconds = min(2 ** attempt_index, 30)
            print(
                f"LLM request failed on attempt {attempt_index}/{LLM_MAX_RETRIES}: "
                f"{request_error!r}. Retrying in {sleep_seconds}s."
            )
            time.sleep(sleep_seconds)

    raise RuntimeError(f"LLM request failed after {LLM_MAX_RETRIES} attempts.") from last_error


## 6. Run Labeling

## Optional Raw-Output Debug Helpers

These helpers mirror the test notebook. They do not run automatically, but they make parser failures auditable by printing the query, candidate documents, raw LLM response, and expected candidate IDs before strict validation.


In [8]:
DEBUG_NUM_QUERY_GROUPS = 3
DEBUG_DOCS_TO_PRINT_PER_QUERY = 12
DEBUG_PRINT_FULL_USER_PROMPT = False


def extract_json_object_from_raw_response(raw_response_text: str) -> dict | None:
    """Best-effort JSON extraction used only for debugging malformed model output."""
    cleaned_response_text = strip_markdown_json_fence(raw_response_text)
    try:
        return json.loads(cleaned_response_text)
    except json.JSONDecodeError:
        pass

    object_start_index = cleaned_response_text.find("{")
    object_end_index = cleaned_response_text.rfind("}")
    if object_start_index == -1 or object_end_index == -1 or object_end_index <= object_start_index:
        return None

    try:
        return json.loads(cleaned_response_text[object_start_index : object_end_index + 1])
    except json.JSONDecodeError:
        return None


def display_debug_query_group_inputs(prompt: dict, docs_to_print_per_query: int | None) -> None:
    """Print the query and candidate documents that will be sent to the LLM."""
    payload = prompt["payload"]
    print("=" * 120)
    print(f"query_id: {payload['query_id']}")
    print(f"query_text: {payload['query_text']}")
    print(f"candidate_count: {len(payload['documents'])}")
    print(f"prompt_version: {prompt['prompt_version']}")

    internal_documents_dataframe = pd.DataFrame(payload["documents"])
    internal_columns_to_show = [
        column_name
        for column_name in [
            "candidate_id",
            "blinded_position",
            "doc_id",
            "title",
            "recipe_type",
            "description",
        ]
        if column_name in internal_documents_dataframe.columns
    ]
    print("\nInternal candidate mapping used by the parser:")
    display(internal_documents_dataframe[internal_columns_to_show].head(docs_to_print_per_query))

    prompt_documents_dataframe = pd.DataFrame(payload["prompt_documents"])
    prompt_columns_to_show = [
        column_name
        for column_name in [
            "candidate_id",
            "title",
            "recipe_type",
            "description",
        ]
        if column_name in prompt_documents_dataframe.columns
    ]
    print("\nCandidate documents sent to the LLM:")
    display(prompt_documents_dataframe[prompt_columns_to_show].head(docs_to_print_per_query))

    if DEBUG_PRINT_FULL_USER_PROMPT:
        print("\nFull user prompt:")
        print(prompt["user_prompt"])


def debug_query_group_llm_outputs(
    query_groups: list[dict],
    num_query_groups: int = DEBUG_NUM_QUERY_GROUPS,
    docs_to_print_per_query: int | None = DEBUG_DOCS_TO_PRINT_PER_QUERY,
    parse_after_response: bool = True,
) -> None:
    """Call the LLM query by query and print raw output before strict parsing."""
    selected_query_groups = query_groups[:num_query_groups]
    print(f"Debug query groups selected: {len(selected_query_groups)}")

    for loop_index, query_group in enumerate(selected_query_groups, start=1):
        prompt = build_relevance_judge_prompt_for_query(query_group)
        payload = prompt["payload"]
        expected_candidate_ids = [document["candidate_id"] for document in payload["documents"]]

        print(f"\nDEBUG CALL {loop_index}/{len(selected_query_groups)}")
        display_debug_query_group_inputs(prompt, docs_to_print_per_query)

        raw_response_text = call_openai_compatible_query_judge(
            prompt["system_prompt"],
            prompt["user_prompt"],
        )

        print("\nRaw LLM response before parsing:")
        print(raw_response_text)

        debug_json_object = extract_json_object_from_raw_response(raw_response_text)
        if debug_json_object is None:
            print("\nDebug JSON extraction: failed to parse any JSON object from the raw response.")
        else:
            print("\nDebug JSON extraction: parsed top-level keys:", list(debug_json_object.keys()))
            debug_labels = debug_json_object.get("labels", [])
            if isinstance(debug_labels, list):
                debug_labels_dataframe = pd.DataFrame(debug_labels)
                print(f"Debug JSON extraction: label rows returned by LLM = {len(debug_labels_dataframe)}")
                display(debug_labels_dataframe.head(docs_to_print_per_query))

        if not parse_after_response:
            continue

        try:
            parsed_label_records = parse_query_label_response(raw_response_text, payload)
        except Exception as parse_error:
            print("\nStrict parser error:")
            print(repr(parse_error))
            print("\nExpected candidate_id values:")
            print(expected_candidate_ids)
            continue

        parsed_labels_dataframe = pd.DataFrame(parsed_label_records)
        print("\nStrict parser succeeded. Parsed labels:")
        display(parsed_labels_dataframe.head(docs_to_print_per_query))


In [9]:

def append_jsonl_records(output_path: Path, records: list[dict]) -> None:
    """Append multiple JSON-serializable records to a JSONL file in one open/flush cycle."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("a", encoding="utf-8") as output_file:
        for record in records:
            output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()


def rewrite_jsonl_records(output_path: Path, records: list[dict]) -> None:
    """Rewrite a JSONL file with the provided records."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as output_file:
        for record in records:
            output_file.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_expected_label_counts_by_query(query_groups: list[dict]) -> dict[int, int]:
    """Return the expected number of labels for each query group."""
    return {
        int(query_group["query_id"]): len(query_group["documents"])
        for query_group in query_groups
    }


def split_complete_and_incomplete_query_ids(
    label_records: list[dict],
    expected_label_counts_by_query: dict[int, int],
) -> tuple[set[int], set[int]]:
    """Classify existing output labels into complete and incomplete query IDs."""
    records_by_query_id: dict[int, list[dict]] = {}
    for record in label_records:
        query_id = int(record["query_id"])
        records_by_query_id.setdefault(query_id, []).append(record)

    complete_query_ids = set()
    incomplete_query_ids = set()

    for query_id, query_records in records_by_query_id.items():
        if query_id not in expected_label_counts_by_query:
            continue

        expected_count = int(expected_label_counts_by_query[query_id])
        candidate_ids = [str(record.get("candidate_id", "")) for record in query_records]
        doc_ids = [int(record["doc_id"]) for record in query_records if "doc_id" in record]
        relevance_values = [int(record["relevance"]) for record in query_records if "relevance" in record]

        is_complete = (
            len(query_records) == expected_count
            and len(set(candidate_ids)) == expected_count
            and len(set(doc_ids)) == expected_count
            and len(relevance_values) == expected_count
            and all(relevance in {0, 1, 2, 3} for relevance in relevance_values)
        )
        if is_complete:
            complete_query_ids.add(query_id)
        else:
            incomplete_query_ids.add(query_id)

    return complete_query_ids, incomplete_query_ids


def load_completed_query_ids_and_cleanup_partial_outputs(
    labels_path: Path,
    query_logs_path: Path,
    expected_label_counts_by_query: dict[int, int],
) -> set[int]:
    """Return completed query IDs and remove partial query outputs before resuming."""
    if not labels_path.exists():
        return set()

    label_records = load_jsonl_records(labels_path)
    complete_query_ids, incomplete_query_ids = split_complete_and_incomplete_query_ids(
        label_records=label_records,
        expected_label_counts_by_query=expected_label_counts_by_query,
    )

    if incomplete_query_ids:
        print(
            "Found incomplete label outputs for query IDs; removing them before resume:",
            sorted(incomplete_query_ids)[:20],
        )
        kept_label_records = [
            record
            for record in label_records
            if int(record["query_id"]) not in incomplete_query_ids
        ]
        rewrite_jsonl_records(labels_path, kept_label_records)

        if query_logs_path.exists():
            query_log_records = load_jsonl_records(query_logs_path)
            kept_log_records = [
                record
                for record in query_log_records
                if int(record["query_id"]) not in incomplete_query_ids
            ]
            rewrite_jsonl_records(query_logs_path, kept_log_records)

    return complete_query_ids


def judge_query_groups_with_llm(
    query_groups: list[dict],
    labels_output_path: Path,
    query_logs_output_path: Path,
    resume: bool,
    max_query_groups: Optional[int] = None,
) -> pd.DataFrame:
    """Label query groups with one LLM call per query and save flattened pair-level labels."""
    selected_query_groups = query_groups[:max_query_groups] if max_query_groups is not None else query_groups
    expected_label_counts_by_query = build_expected_label_counts_by_query(selected_query_groups)

    if not resume:
        for output_path in [labels_output_path, query_logs_output_path]:
            if output_path.exists():
                output_path.unlink()

    completed_query_ids = (
        load_completed_query_ids_and_cleanup_partial_outputs(
            labels_path=labels_output_path,
            query_logs_path=query_logs_output_path,
            expected_label_counts_by_query=expected_label_counts_by_query,
        )
        if resume
        else set()
    )
    saved_label_records = []

    print(f"Query groups selected: {len(selected_query_groups)}")
    print(f"Already completed query IDs: {len(completed_query_ids)}")
    print(f"Remaining query IDs to label: {len(selected_query_groups) - len(completed_query_ids)}")

    for index, query_group in enumerate(selected_query_groups, start=1):
        query_id = int(query_group["query_id"])
        if query_id in completed_query_ids:
            continue

        prompt = build_relevance_judge_prompt_for_query(query_group)
        raw_response_text = call_openai_compatible_query_judge(prompt["system_prompt"], prompt["user_prompt"])
        try:
            parsed_label_records = parse_query_label_response(raw_response_text, prompt["payload"])
        except Exception as parse_error:
            print("\nStrict parser failed inside the main labeling runner.")
            display_debug_query_group_inputs(prompt, DEBUG_DOCS_TO_PRINT_PER_QUERY)
            print("\nRaw LLM response before parsing:")
            print(raw_response_text)
            print("\nStrict parser error:")
            print(repr(parse_error))
            raise

        output_records = [
            {
                **label_record,
                "judge_type": "llm",
                "llm_provider": "openai_compatible_litellm",
                "llm_model_name": LLM_MODEL_NAME,
                "prompt_version": prompt["prompt_version"],
            }
            for label_record in parsed_label_records
        ]

        append_jsonl_records(labels_output_path, output_records)
        saved_label_records.extend(output_records)

        query_log_record = {
            "query_id": query_id,
            "judge_type": "llm",
            "llm_provider": "openai_compatible_litellm",
            "llm_model_name": LLM_MODEL_NAME,
            "prompt_version": prompt["prompt_version"],
            "candidate_count": len(prompt["payload"]["documents"]),
            "parsed_label_count": len(parsed_label_records),
            "raw_response_text": raw_response_text,
        }
        append_jsonl_record(query_logs_output_path, query_log_record)

        completed_query_ids.add(query_id)
        print(
            f"[{index}/{len(selected_query_groups)}] query_id={query_id} "
            f"saved {len(parsed_label_records)} labels"
        )

    return pd.DataFrame(saved_label_records)


In [10]:
# This is the main labeling run.
# It performs one OpenAI-compatible LiteLLM call per query group.
llm_label_dataframe = judge_query_groups_with_llm(
    query_groups=query_groups,
    labels_output_path=LLM_LABELS_PATH,
    query_logs_output_path=LLM_QUERY_LOGS_PATH,
    resume=RESUME_LLM_LABELING,
    max_query_groups=MAX_QUERY_GROUPS_TO_LABEL,
)
llm_label_dataframe.head()

Query groups selected: 500
Already completed query IDs: 431
Remaining query IDs to label: 69
[432/500] query_id=431 saved 50 labels
[433/500] query_id=432 saved 50 labels
[434/500] query_id=433 saved 50 labels
[435/500] query_id=434 saved 50 labels
[436/500] query_id=435 saved 50 labels
[437/500] query_id=436 saved 50 labels
[438/500] query_id=437 saved 50 labels
[439/500] query_id=438 saved 50 labels
[440/500] query_id=439 saved 50 labels
[441/500] query_id=440 saved 50 labels
[442/500] query_id=441 saved 50 labels
[443/500] query_id=442 saved 50 labels
[444/500] query_id=443 saved 50 labels
[445/500] query_id=444 saved 50 labels
[446/500] query_id=445 saved 50 labels
[447/500] query_id=446 saved 50 labels
[448/500] query_id=447 saved 50 labels
[449/500] query_id=448 saved 50 labels
[450/500] query_id=449 saved 50 labels
[451/500] query_id=450 saved 50 labels
[452/500] query_id=451 saved 50 labels
[453/500] query_id=452 saved 50 labels
[454/500] query_id=453 saved 50 labels
[455/500] 

,query_id,candidate_id,doc_id,blinded_position,relevance,label_match_strategy,label_reconciled,judge_type,llm_provider,llm_model_name,prompt_version
0,431,D001,9608,1,0,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
1,431,D002,9878,2,2,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
2,431,D003,8638,3,0,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
3,431,D004,3548,4,1,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...
4,431,D005,681,5,1,candidate_id,False,llm,openai_compatible_litellm,local-std-03,recipe_relevance_query_batch_title_type_descri...


## 7. Summarize Labels

In [12]:
def summarize_llm_groundtruth_labels(llm_labels_path: Path) -> pd.DataFrame:
    """Summarize LLM-generated relevance labels before human validation."""
    llm_records = load_jsonl_records(llm_labels_path)
    label_dataframe = pd.DataFrame(llm_records)
    if label_dataframe.empty:
        raise ValueError(f"No labels found in {llm_labels_path}")

    print("Number of judged pairs:", len(label_dataframe))
    print("Number of queries:", label_dataframe["query_id"].nunique())
    print("Number of documents:", label_dataframe["doc_id"].nunique())
    print("\nOverall relevance distribution:")
    print(label_dataframe["relevance"].value_counts().sort_index())
    print("\nJudged pairs per query:")
    print(label_dataframe.groupby("query_id").size().describe())
    return label_dataframe


# Run after labeling completes.
llm_groundtruth_dataframe = summarize_llm_groundtruth_labels(LLM_LABELS_PATH)

Number of judged pairs: 25000
Number of queries: 500
Number of documents: 7571

Overall relevance distribution:
relevance
0    13505
1     7713
2     3013
3      769
Name: count, dtype: int64

Judged pairs per query:
count    500.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
dtype: float64
